In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = sns.load_dataset('taxis')
df = df.dropna(subset=['payment', 'pickup_borough', 'dropoff_borough'])
df['pickup'] = pd.to_datetime(df['pickup'])

target = (df['payment'] == 'credit card').astype(int)
df.head()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,2.15,0.0,12.95,yellow,credit card,Lenox Hill West,UN/Turtle Bay South,Manhattan,Manhattan
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,0.00,0.0,9.30,yellow,cash,Upper West Side South,Upper West Side South,Manhattan,Manhattan
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,2.36,0.0,14.16,yellow,credit card,Alphabet City,West Village,Manhattan,Manhattan
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,6.15,0.0,36.95,yellow,credit card,Hudson Sq,Yorkville West,Manhattan,Manhattan
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,1.10,0.0,13.40,yellow,credit card,Midtown East,Yorkville West,Manhattan,Manhattan


In [2]:
baseline_numeric = ['passengers', 'distance', 'fare', 'tolls']
baseline_categorical = ['pickup_borough', 'dropoff_borough']

X_baseline = df[baseline_numeric + baseline_categorical].copy()
X_baseline.head()

,passengers,distance,fare,tolls,pickup_borough,dropoff_borough
0,1,1.60,7.0,0.0,Manhattan,Manhattan
1,1,0.79,5.0,0.0,Manhattan,Manhattan
2,1,1.37,7.5,0.0,Manhattan,Manhattan
3,1,7.70,27.0,0.0,Manhattan,Manhattan
4,3,2.16,9.0,0.0,Manhattan,Manhattan


In [3]:
engineered = df.copy()

# 1-3: datetime-derived features extracted from the pickup timestamp
engineered['pickup_hour'] = engineered['pickup'].dt.hour
engineered['pickup_dayofweek'] = engineered['pickup'].dt.dayofweek
engineered['is_weekend'] = engineered['pickup_dayofweek'].isin([5, 6]).astype(int)

# 4: interaction feature (ratio) - fare charged per mile travelled
engineered['fare_per_mile'] = engineered['fare'] / engineered['distance'].replace(0, np.nan)
engineered['fare_per_mile'] = engineered['fare_per_mile'].fillna(engineered['fare_per_mile'].median())

# 5: interaction feature (multiply) - combined effect of trip length and party size
engineered['distance_passenger_interaction'] = engineered['distance'] * engineered['passengers']

# 6: log transform of a right-skewed numeric column (distance, skew ~ 3.0)
engineered['log_distance'] = np.log1p(engineered['distance'])

# 7: binned version of a continuous variable (trip distance -> short/medium/long)
engineered['distance_bin'] = pd.cut(
    engineered['distance'], bins=[-0.01, 1, 3, engineered['distance'].max()],
    labels=['short', 'medium', 'long']
)

engineered[['pickup_hour', 'pickup_dayofweek', 'is_weekend', 'fare_per_mile',
            'distance_passenger_interaction', 'log_distance', 'distance_bin']].head()

,pickup_hour,pickup_dayofweek,is_weekend,fare_per_mile,distance_passenger_interaction,log_distance,distance_bin
0,20,5,1,4.375000,1.60,0.955511,medium
1,16,0,0,6.329114,0.79,0.582216,short
2,17,2,0,5.474453,1.37,0.862890,medium
3,1,6,1,3.506494,7.70,2.163323,long
4,13,5,1,4.166667,6.48,1.150572,medium


In [4]:
engineered_numeric = baseline_numeric + [
    'pickup_hour', 'pickup_dayofweek', 'is_weekend',
    'fare_per_mile', 'distance_passenger_interaction', 'log_distance'
]
engineered_categorical = baseline_categorical + ['distance_bin']

X_engineered = engineered[engineered_numeric + engineered_categorical].copy()
X_engineered.head()

,passengers,distance,fare,tolls,pickup_hour,pickup_dayofweek,is_weekend,fare_per_mile,distance_passenger_interaction,log_distance,pickup_borough,dropoff_borough,distance_bin
0,1,1.60,7.0,0.0,20,5,1,4.375000,1.60,0.955511,Manhattan,Manhattan,medium
1,1,0.79,5.0,0.0,16,0,0,6.329114,0.79,0.582216,Manhattan,Manhattan,short
2,1,1.37,7.5,0.0,17,2,0,5.474453,1.37,0.862890,Manhattan,Manhattan,medium
3,1,7.70,27.0,0.0,1,6,1,3.506494,7.70,2.163323,Manhattan,Manhattan,long
4,3,2.16,9.0,0.0,13,5,1,4.166667,6.48,1.150572,Manhattan,Manhattan,medium


In [5]:
def evaluate(X, numeric_cols, categorical_cols, y, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocessor = ColumnTransformer([
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', RandomForestClassifier(n_estimators=200, random_state=42))
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    acc = accuracy_score(y_test, preds)
    n_features = pipeline.named_steps['preprocess'].transform(X_train).shape[1]
    print(f'{label}: accuracy = {acc:.4f}, encoded feature count = {n_features}')
    return acc

acc_before = evaluate(X_baseline, baseline_numeric, baseline_categorical, target, 'Before feature engineering')
acc_after = evaluate(X_engineered, engineered_numeric, engineered_categorical, target, 'After feature engineering')

print(f'\nAccuracy delta: {acc_after - acc_before:+.4f}')

Before feature engineering: accuracy = 0.6667, encoded feature count = 13


After feature engineering: accuracy = 0.6958, encoded feature count = 22

Accuracy delta: +0.0292
